In [1]:
%pip install gwpy gwosc scipy # to get noise data
%pip install pycbc # to generate waveforms

  Using cached gwpy-4.0.2-py3-none-any.whl.metadata (4.9 kB)
  Using cached gwosc-0.8.3-py3-none-any.whl.metadata (6.1 kB)
  Using cached astropy-8.0.1-cp311-abi3-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached dateparser-1.4.3-py3-none-any.whl.metadata (25 kB)
  Using cached dqsegdb2-1.3.1-py3-none-any.whl.metadata (3.1 kB)
  Using cached gwdatafind-2.1.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached ligotimegps-2.1.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached pyerfa-2.0.1.5-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.7 kB)
  Using cached tzlocal-5.4.4-py3-none-any.whl.metadata (7.7 kB)
  Using cached igwn_auth_utils-1.4.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached safe_netrc-1.0.1-py3-none-any.whl.metadata (1.9 kB)
  Using cached scitokens-1.9.7-py3-none-any.whl.metadata (12 kB)
Using cached gwpy-4.0.2-py3-none-any.whl (1.6 MB)
Using cached gwosc-0.8.3-py3-none-any.whl (36 kB)
   ━━━━━━━━━━━━━━━━

the above lines install the necessary packages to access the noise time series data and the gw signal generator function.

the code below imports the necessary functions/libraries and finds the start and end times of the 03a run and 03b runs. it uses these times to retreive and print all the real gw events recorded during those runs. as shown in the output below, many events were recorded, so clean noise segments can not be determined manually.

In [2]:
from gwpy.timeseries import TimeSeries
from gwosc.datasets import event_gps
from gwosc.datasets import find_datasets, run_segment

# get 03a, set 03a boundaries
start_o3a, end_o3a = run_segment("O3a")
print("03a [sec] start - end : " ,start_o3a, end_o3a)
print()
# all confirmed 03a event names
events_o3a = find_datasets(type="event", segment=(start_o3a, end_o3a))
# event times
event_times_03a = [event_gps(name) for name in events_o3a]
#print(event_times_03a)
#print()

# get 03b, set 03b boundaries
start_o3b, end_o3b = run_segment("O3b")
print("03b [sec] start - end : " , start_o3b, end_o3b)
print()
# all confirmed 03b event names
events_o3b = find_datasets(type="event", segment=(start_o3b, end_o3b))
# event times
event_times_03b = [event_gps(name) for name in events_o3b]
#print(event_times_03b)

/Users/roxydhyana/miniconda3/envs/signal/lib/python3.14/site-packages/gwpy/time/_ligotimegps.py:42: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


03a [sec] start - end :  1238166018 1253977218

03b [sec] start - end :  1256655618 1269363618



the two functions below analytically determine clean noise segments consistent across all 3 detectors.

the code below defines the function find_clean_segments which identifies n candidate segments within a given observing run that do not overlap with any confirmed GW event. the function all_detectors_online checks each candidate segment against all 3 detectors, keeping only the segments where H1, L1, and V1 were all simultaneously recording data.

In [3]:
from gwosc.timeline import get_segments

def find_clean_segments(start, end, duration, event_times, buffer=60, n_segments=12):
    segments = []
    t = start
    spacing = (end-start) // (n_segments)
    while len(segments) < n_segments and t + duration < end:
        segment_end = t + duration
        buffer_start = t - buffer
        buffer_end = segment_end + buffer
        if all(not (buffer_start < event < buffer_end) for event in event_times):
            segments.append((t, segment_end))
            t += spacing  # move past this window
        else:
            t += 60  # skip ahead and try again
    return segments

def all_detectors_online(start, end):
    for ifo in ["H1", "L1", "V1"]:
        segs = get_segments(f"{ifo}_DATA", start, end)
        if not segs or segs[0] != (start, end):
            return False
    return True

def identify_usable_segments (noise_data, noise_array, clean_noise):
    count = 0
    for start, end in clean_noise:  
        if all_detectors_online(start, end):
            for ifo in ["H1", "L1", "V1"]:
                data = TimeSeries.fetch_open_data(ifo, start, end, cache=True)
                noise_data[ifo].append(data)
                noise_array[ifo].append(data.value)
            print(f"Using {start}-{end}")
            count+=1
        else:
            print(f"Skipping {start}-{end}, not all detectors online")

    total_segments = len(noise_data["H1"])
    total_seconds = total_segments * 300
    print()
    print(f"Total segments all detectors online: {count}")
    print(f"Total seconds: {total_seconds}")

clean_noise_o3a = find_clean_segments(start_o3a, end_o3a, 300, event_times_03a)

noise_data_o3a = {"H1": [], "L1": [], "V1": []}
noise_array_o3a = {"H1": [], "L1": [], "V1": []} # gwpy TimeSeries -> numpy array

identify_usable_segments (noise_data_o3a, noise_array_o3a, clean_noise_o3a)

Using 1238166018-1238166318
Skipping 1239483618-1239483918, not all detectors online
Using 1240801218-1240801518
Skipping 1242118818-1242119118, not all detectors online
Using 1243436418-1243436718
Using 1244754018-1244754318
Skipping 1246071618-1246071918, not all detectors online
Using 1247389218-1247389518
Using 1248706818-1248707118
Skipping 1250024418-1250024718, not all detectors online
Using 1251342018-1251342318
Using 1252659618-1252659918

Total segments all detectors online: 8
Total seconds: 2400


In [4]:
clean_noise_o3b = find_clean_segments(start_o3b, end_o3b, 300, event_times_03b, n_segments=14)

noise_data_o3b = {"H1": [], "L1": [], "V1": []}
noise_array_o3b = {"H1": [], "L1": [], "V1": []} # gwpy TimeSeries -> numpy array

identify_usable_segments (noise_data_o3b, noise_array_o3b, clean_noise_o3b)

Skipping 1256655618-1256655918, not all detectors online
Using 1257563332-1257563632
Using 1258471046-1258471346
Using 1259378760-1259379060
Skipping 1260286474-1260286774, not all detectors online
Using 1261194188-1261194488
Using 1262101902-1262102202
Using 1263009616-1263009916
Using 1263917330-1263917630
Skipping 1264825044-1264825344, not all detectors online
Using 1265732758-1265733058
Using 1266640472-1266640772
Skipping 1267548186-1267548486, not all detectors online
Using 1268455900-1268456200

Total segments all detectors online: 10
Total seconds: 3000
